<a href="https://colab.research.google.com/github/Born3Life/234-tkst-bots/blob/main/ollama_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ollama + Llava на Google Colab
Запускаем Ollama с моделью llava и открываем доступ через туннель Cloudflare.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq wget curl zstd

!curl -fsSL https://ollama.com/install.sh | sh

import os
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_KEEP_ALIVE'] = '24h'

get_ipython().system_raw('ollama serve &')
import time
time.sleep(5)
print('✅ Ollama запущен')

In [ ]:
!ollama pull llava
!ollama pull gemma3:12b
print('✅ Модели llava (зрение) и gemma3:12b (текст) готовы')

In [ ]:
import subprocess
import threading
import re
import time

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

found_url = []

def run_tunnel():
    proc = subprocess.Popen(
        ['./cloudflared', 'tunnel', '--url', 'http://localhost:11434'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        print(line, end='')
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            url = f"{match.group()}/api/chat"
            found_url.append(url)
            print(f'\n\n*** МОЙ URL ***')
            print(f'{url}')
            print(f'***************\n')

thread = threading.Thread(target=run_tunnel, daemon=True)
thread.start()

print('⏳ Ожидаю туннель... (до 30 сек)')
for _ in range(30):
    if found_url:
        break
    time.sleep(1)

if found_url:
    print(f'\n✅ Туннель готов: {found_url[0]}')
else:
    print('\n⚠️ Туннель не появился за 30 сек. Проверь вывод выше.')

In [ ]:
import threading
import requests

def keep_alive():
    while True:
        time.sleep(60)
        try:
            requests.get(f'http://localhost:11434', timeout=5)
        except:
            pass

ka = threading.Thread(target=keep_alive, daemon=True)
ka.start()
print('✅ Keep-alive запущен (пинг каждые 60 сек)')

## Инструкция для ИИ-ассистента

Colab запущен и работает. Чтобы подключить ботов к этому туннелю:

1. Скопировать URL вида `https://что-то.trycloudflare.com/api/chat`
2. Открыть `bot/services/openrouter.py` в каждом из 4 ботов
3. Заменить `API_URL` на этот URL
4. Поменять модель на `llava` (если не стоит)
5. Закоммитить и запушить на GitHub
6. Render перезапустит ботов автоматически (~2 мин)

---
Colab живёт ~12 часов. Для продления — перезапустить ячейки 1-4.